# 🎞️🎞️ recommendation-system-netflix-amazon-style

## 1) Problem Defination

Modern digital platforms such as streaming services and e-commerce stores must recommend items that match a user’s preferences to increase engagement, satisfaction, and revenue. The goal of this project is to build a Personalized Recommendation System that predicts how much a user will like a movie they have not watched yet and recommends the most relevant items.

This system will replicate the recommendation strategies used by companies like Netflix and Amazon by combining:

  ✔Collaborative Filtering (learning from user–item interactions)

  ✔Content-Based Filtering (learning from item metadata)

  ✔Hybrid Recommendation (combining scores from both methods)

The final system should be able to:

✔ Generate personalized movie recommendations

✔ Recommend similar movies to a selected movie

✔ Provide ranking-quality predictions using multiple evaluation metrics

# 2) data

The MovieLens 100K dataset is a widely used benchmark dataset in the field of recommender systems. It was collected by the GroupLens Research Group and contains 100,000 ratings from 943 users on 1,682 movies. Each user has rated at least 20 movies, ensuring sufficient interaction data for building recommendation models.

This dataset is specifically designed for evaluating collaborative filtering and recommendation algorithms.
https://grouplens.org/datasets/movielens/100k/

# 3) Evaluation data

The evaluation of the recommendation system is performed using the MovieLens 100K dataset obtained from the official GroupLens repository. The dataset is split into training and testing subsets to assess the model’s ability to generalize to unseen user–item interactions.
Train-Test Split

The dataset is divided as follows:

80% Training Data

Used to train the recommendation models (Collaborative Filtering, Content-Based, Hybrid)
20% Testing Data
Used to evaluate model performance on unseen data
To ensure consistency:
The split is done randomly
Each user appears in both training and testing sets
This avoids the cold-start issue during evaluation


The dataset is split into training (80%) and testing (20%) sets. The model is evaluated using both rating prediction metrics (RMSE, MAE) and ranking metrics (Precision@K, Recall@K, NDCG). This ensures a comprehensive evaluation of both prediction accuracy and recommendation quality.


## start from the Libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

In [3]:
#importing data from the drive
ratings  = pd.read_csv("/content/drive/MyDrive/ml-100k/u.data",sep="\t",names=["user_id","item_id","rating","timestamp"])
movies = pd.read_csv("/content/drive/MyDrive/ml-100k/u.item", sep="|", encoding="latin-1", header=None)
movies = movies[[0, 1, 2]]  # id, title, release date
movies.columns = ["movie_id", "title", "release_date"]

print(ratings.head())
print(movies.head())

   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596
   movie_id              title release_date
0         1   Toy Story (1995)  01-Jan-1995
1         2   GoldenEye (1995)  01-Jan-1995
2         3  Four Rooms (1995)  01-Jan-1995
3         4  Get Shorty (1995)  01-Jan-1995
4         5     Copycat (1995)  01-Jan-1995


## Train-Test Split

In [4]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)
rating_count = train_data.groupby("item_id").size()

In [5]:
print("Train size:", train_data.shape)
print("Test size:", test_data.shape)

Train size: (80000, 4)
Test size: (20000, 4)


## Creating user-item matrix(collaborative filtering)

In [6]:
user_item_matrix = train_data.pivot_table(index='user_id', columns='item_id', values='rating',)
print(user_item_matrix.shape)
user_item_matrix.head()

(943, 1653)


item_id,1,2,3,4,5,6,7,8,9,10,...,1668,1670,1671,1672,1673,1676,1678,1679,1680,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,3.0,4.0,NaN,3.0,NaN,4.0,NaN,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
user_item_filled = user_item_matrix.fillna(0)

In [8]:
user_item_filled.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1668,1670,1671,1672,1673,1676,1678,1679,1680,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,3.0,4.0,0.0,3.0,0.0,4.0,0.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## user-user similarity(collaborative filtering)

In [9]:
user_similarity = cosine_similarity(user_item_filled)

user_similarity_df = pd.DataFrame(user_similarity,
                                  index= user_item_filled.index,
                                  columns=user_item_filled.index)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.136196,0.030424,0.026203,0.284613,0.331412,0.319056,0.274139,0.083486,0.281396,...,0.277459,0.084849,0.205849,0.144161,0.133679,0.092367,0.216948,0.084181,0.104599,0.329288
2,0.136196,1.000000,0.114644,0.168220,0.093128,0.162165,0.095848,0.091360,0.149476,0.125701,...,0.149359,0.268977,0.320095,0.323347,0.241012,0.152655,0.230951,0.117484,0.166632,0.096719
3,0.030424,0.114644,1.000000,0.346894,0.000000,0.085071,0.032829,0.053875,0.060177,0.052552,...,0.021713,0.017707,0.154299,0.049358,0.107604,0.019022,0.101207,0.021959,0.127179,0.013805
4,0.026203,0.168220,0.346894,1.000000,0.011848,0.051287,0.075209,0.142100,0.060465,0.035202,...,0.034908,0.044480,0.087428,0.118082,0.100612,0.000000,0.151086,0.110324,0.112342,0.032367
5,0.284613,0.093128,0.000000,0.011848,1.000000,0.168527,0.298438,0.185290,0.039737,0.166013,...,0.276012,0.103529,0.085547,0.072429,0.104445,0.049198,0.204472,0.148028,0.099978,0.247527


User-Based Collaborative Filtering Prediction Function
Idea (important intuition)

To predict how a user u will rate a movie i:

We use ratings from similar users who already rated that movie.


In [10]:
def clip_rating(r):
    return max(1, min(5, r))

def predict_rating(user_id, item_id, user_item_matrix, user_similarity_df, k=10):
    """
    Predict rating for a given user and item using User-Based Collaborative Filtering
    """

    # If item not in training data
    if item_id not in user_item_matrix.columns:
        return np.nan

    user_mean = user_item_matrix.loc[user_id].mean()

    similar_users = user_similarity_df[user_id].drop(user_id)
    similar_users = similar_users[similar_users > 0.1].head(k)

    numerator = 0
    denominator = 0
    support = 0

    for sim_user, sim_score in similar_users.items():

        rating = user_item_matrix.loc[sim_user, item_id]

        if np.isnan(rating):
            continue

        sim_user_mean = user_item_matrix.loc[sim_user].mean()

        numerator += sim_score * (rating - sim_user_mean)
        denominator += abs(sim_score)
        support += 1

    if support < 3 or denominator == 0:
        return clip_rating(user_mean)

    pred = user_mean + (numerator / denominator)

    return clip_rating(pred)

# test the function

In [11]:
user_id = 1
item_id = 50

pred = predict_rating(
    user_id,
    item_id,
    user_item_matrix,
    user_similarity_df,
    k=10
)

print("Predicted rating:", pred)

Predicted rating: 4.387438362149823


## Top-N Recommendation Function

The idea is:-

We Look at all movies the user has NOT rated

Predict rating for each one

Sort predictions

Return top N movie

In [12]:
def get_top_n_recommendations(user_id, user_item_matrix, user_similarity_df, movies_df, n=10):

    all_items = user_item_matrix.columns
    rated_items = user_item_matrix.loc[user_id].dropna().index.tolist()
    unrated_items = [item for item in all_items if item not in rated_items]

    predictions = []

    for item_id in unrated_items:

        # 🔧 FIX: use predict_rating() (your actual function)
        pred_rating = predict_rating(
            user_id,
            item_id,
            user_item_matrix,
            user_similarity_df,
            k=10
        )

        if np.isnan(pred_rating):
            continue

        # Popularity boost
        pop_score = rating_count.get(item_id, 0) / rating_count.max()

        final_score = 0.7 * pred_rating + 0.3 * pop_score

        predictions.append((item_id, final_score))

    predictions.sort(key=lambda x: x[1], reverse=True)

    top_n = predictions[:n]

    results = []
    for item_id, score in top_n:
        title = movies_df[movies_df["movie_id"] == item_id]["title"].values
        if len(title) > 0:
            results.append((title[0], score))

    return results

In [13]:
user_id = 1

recommendations = get_top_n_recommendations(
    user_id,
    user_item_matrix,
    user_similarity_df,
    movies,
    n=10
)

for i, (title, score) in enumerate(recommendations, 1):
    print(f"{i}. {title} — score: {score:.2f}")

1. Fargo (1996) — score: 3.74
2. Stand by Me (1986) — score: 3.61
3. Heathers (1989) — score: 3.59
4. Glory (1989) — score: 3.59
5. Maltese Falcon, The (1941) — score: 3.57
6. Dumbo (1941) — score: 3.56
7. Jerry Maguire (1996) — score: 3.55
8. Return of the Jedi (1983) — score: 3.47
9. E.T. the Extra-Terrestrial (1982) — score: 3.43
10. Butch Cassidy and the Sundance Kid (1969) — score: 3.42


## Evaluation

In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_rating_predictions(test_data, user_item_matrix, user_similarity_df):
    y_true = []
    y_pred = []

    for _, row in test_data.iterrows():
        user = row['user_id']
        item = row['item_id']
        true_rating = row['rating']

        # Only evaluate for users & items seen in training
        if user in user_item_matrix.index and item in user_item_matrix.columns:

            pred = predict_rating(user, item, user_item_matrix, user_similarity_df)

            if not np.isnan(pred):
                y_true.append(true_rating)
                y_pred.append(pred)

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae

rmse, mae = evaluate_rating_predictions(test_data, user_item_matrix, user_similarity_df)

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 1.0645832304959681
MAE: 0.8393212132836374


In [15]:
def get_top_n_item_ids(user_id, user_item_matrix, user_similarity_df, n=10):
    all_items = user_item_matrix.columns
    rated_items = user_item_matrix.loc[user_id].dropna().index.tolist()
    unrated_items = [item for item in all_items if item not in rated_items]

    predictions = []

    for item in unrated_items:
        pred = predict_rating(user_id, item, user_item_matrix, user_similarity_df)

        if not np.isnan(pred):
            predictions.append((item, pred))

    predictions.sort(key=lambda x: x[1], reverse=True)
    top_n = [item for item, score in predictions[:n]]

    return top_n

In [16]:
def precision_at_k(recommended_items, relevant_items, k):
    hit_count = len(set(recommended_items[:k]) & set(relevant_items))
    return hit_count / k

def recall_at_k(recommended_items, relevant_items, k):
    if len(relevant_items) == 0:
        return 0
    hit_count = len(set(recommended_items[:k]) & set(relevant_items))
    return hit_count / len(relevant_items)

def ndcg_at_k(recommended_items, relevant_items, k):
    dcg = 0
    for i, item in enumerate(recommended_items[:k]):
        if item in relevant_items:
            dcg += 1 / math.log2(i + 2)

    idcg = sum([1 / math.log2(i + 2) for i in range(min(k, len(relevant_items)))])

    return dcg / idcg if idcg > 0 else 0

In [17]:
def evaluate_ranking(test_data, user_item_matrix, user_similarity_df, k=10):
    precision_list = []
    recall_list = []
    ndcg_list = []

    for user in test_data['user_id'].unique():

        if user not in user_item_matrix.index:
            continue

        # Items the user liked in test set (ground truth)
        user_test = test_data[test_data['user_id'] == user]
        relevant_items = user_test[user_test['rating'] >= 4]['item_id'].tolist()

        if len(relevant_items) == 0:
            continue

        recommended_items = get_top_n_item_ids(user, user_item_matrix, user_similarity_df, n=k)

        precision_list.append(precision_at_k(recommended_items, relevant_items, k))
        recall_list.append(recall_at_k(recommended_items, relevant_items, k))
        ndcg_list.append(ndcg_at_k(recommended_items, relevant_items, k))

    return (
        np.mean(precision_list),
        np.mean(recall_list),
        np.mean(ndcg_list),
    )

precision, recall, ndcg = evaluate_ranking(
    test_data,
    user_item_matrix,
    user_similarity_df,
    k=10
)

print("Precision@10:", precision)
print("Recall@10:", recall)
print("NDCG@10:", ndcg)

Precision@10: 0.07108695652173913
Recall@10: 0.05841313311944579
NDCG@10: 0.0807124319293662
